# Train ResNet18 ? Brain MRI Classification

This notebook trains ResNet18 and saves the best checkpoint. **Training only happens when you run cells in this notebook.**

**Prerequisites**
- Run the data-preparation notebook first.
- It must have created `prepared_data/train.csv`, `prepared_data/validation.csv`, and `prepared_data/test.csv`.
- The CSVs use the same fixed splits as the EfficientNetV2-S notebooks.


## 1. Imports and configuration


In [1]:
# %pip install pandas pillow matplotlib scikit-learn torch torchvision
from pathlib import Path
import copy, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DATA_DIR=Path("prepared_data"); MODEL_DIR=Path("trained_models"); MODEL_DIR.mkdir(exist_ok=True)
TRAIN_CSV=DATA_DIR/"train.csv"; VAL_CSV=DATA_DIR/"validation.csv"; TEST_CSV=DATA_DIR/"test.csv"
CHECKPOINT_PATH=MODEL_DIR/"resnet18_best.pth"; HISTORY_PATH=MODEL_DIR/"resnet18_training_history.csv"
PREDICTIONS_PATH=MODEL_DIR/"resnet18_test_predictions.csv"; COMPARISON_PATH=MODEL_DIR/"model_comparison.csv"
BATCH_SIZE=32; HEAD_EPOCHS=3; FINETUNE_EPOCHS=7
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:",device)


c:\Users\rickv\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\rickv\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Device: cuda


## 2. Load the saved splits

This notebook does not recreate or change the split. It uses the CSVs saved during data preparation.


In [2]:
for csv_path in [TRAIN_CSV,VAL_CSV,TEST_CSV]:
    if not csv_path.exists(): raise FileNotFoundError(f"Missing {csv_path}. Run data preparation first.")
train_df=pd.read_csv(TRAIN_CSV); val_df=pd.read_csv(VAL_CSV); test_df=pd.read_csv(TEST_CSV)
required={"filepath","class"}
for name,frame in [("train",train_df),("validation",val_df),("test",test_df)]:
    missing=required-set(frame.columns)
    if missing: raise ValueError(f"{name}.csv is missing columns: {sorted(missing)}")
class_names=sorted(train_df["class"].unique().tolist())
class_to_idx={name:i for i,name in enumerate(class_names)}; idx_to_class={i:name for name,i in class_to_idx.items()}
for name,frame in [("validation",val_df),("test",test_df)]:
    unknown=set(frame["class"].unique())-set(class_names)
    if unknown: raise ValueError(f"Unexpected classes in {name}: {sorted(unknown)}")
print("Classes:",class_to_idx); print(f"Train: {len(train_df)} | Validation: {len(val_df)} | Test: {len(test_df)}")
display(pd.DataFrame({"train":train_df["class"].value_counts(),"validation":val_df["class"].value_counts(),"test":test_df["class"].value_counts()}).fillna(0).astype(int).sort_index())


Classes: {'glioma_tumor': 0, 'meningioma_tumor': 1, 'normal': 2, 'pituitary_tumor': 3}
Train: 1991 | Validation: 427 | Test: 427


,train,validation,test
class,,,
glioma_tumor,619,133,133
meningioma_tumor,571,122,123
normal,217,47,46
pituitary_tumor,584,125,125


## 3. Create datasets and dataloaders

ResNet18 uses its standard 224 ? 224 pretrained-input transform.


In [3]:
weights=ResNet18_Weights.DEFAULT
train_transform=transforms.Compose([transforms.Resize((224,224)),transforms.RandomRotation(5),transforms.ToTensor(),transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
eval_transform=weights.transforms()
class BrainMRIDataset(Dataset):
    def __init__(self,dataframe,transform=None): self.dataframe=dataframe.reset_index(drop=True).copy(); self.transform=transform
    def __len__(self): return len(self.dataframe)
    def __getitem__(self,index):
        row=self.dataframe.iloc[index]
        with Image.open(row["filepath"]) as img: image=img.convert("RGB")
        if self.transform: image=self.transform(image)
        return image,class_to_idx[row["class"]]
train_dataset=BrainMRIDataset(train_df,train_transform); val_dataset=BrainMRIDataset(val_df,eval_transform); test_dataset=BrainMRIDataset(test_df,eval_transform)
generator=torch.Generator().manual_seed(SEED)
train_loader=DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True,num_workers=0,generator=generator)
val_loader=DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0); test_loader=DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0)
print("Training batch shape:",tuple(next(iter(train_loader))[0].shape))


Training batch shape: (32, 3, 224, 224)


## 4. Build and train ResNet18

The classifier is trained first with the feature extractor frozen, followed by full-network fine-tuning. The best validation-loss checkpoint is saved with its class metadata.


In [4]:
model=resnet18(weights=weights); model.fc=nn.Linear(model.fc.in_features,len(class_names)); model=model.to(device); criterion=nn.CrossEntropyLoss()
history=[]; best_val_loss=float("inf")
def run_phase(optimizer,epochs,name):
    global best_val_loss
    for epoch in range(epochs):
        model.train(); train_loss=train_correct=train_count=0
        for images,labels in train_loader:
            images,labels=images.to(device),labels.to(device); optimizer.zero_grad(); outputs=model(images); loss=criterion(outputs,labels); loss.backward(); optimizer.step()
            train_loss+=loss.item()*labels.size(0); train_correct+=(outputs.argmax(1)==labels).sum().item(); train_count+=labels.size(0)
        model.eval(); val_loss=val_correct=val_count=0
        with torch.no_grad():
            for images,labels in val_loader:
                images,labels=images.to(device),labels.to(device); outputs=model(images); loss=criterion(outputs,labels); val_loss+=loss.item()*labels.size(0); val_correct+=(outputs.argmax(1)==labels).sum().item(); val_count+=labels.size(0)
        train_loss/=train_count; train_acc=train_correct/train_count; val_loss/=val_count; val_acc=val_correct/val_count
        if val_loss<best_val_loss:
            best_val_loss=val_loss
            torch.save({"model_state_dict":model.state_dict(),"class_names":class_names,"class_to_idx":class_to_idx,"architecture":"resnet18","input_size":224,"best_val_loss":best_val_loss,"seed":SEED},CHECKPOINT_PATH)
        history.append({"phase":name,"epoch":epoch+1,"train_loss":train_loss,"train_accuracy":train_acc,"val_loss":val_loss,"val_accuracy":val_acc})
        print(f"{name} | Epoch {epoch+1}/{epochs} | train {train_loss:.4f}/{train_acc:.4f} | val {val_loss:.4f}/{val_acc:.4f}")
for p in model.parameters(): p.requires_grad=False
for p in model.fc.parameters(): p.requires_grad=True
run_phase(optim.AdamW(model.fc.parameters(),lr=1e-3,weight_decay=1e-4),HEAD_EPOCHS,"Frozen backbone")
for p in model.parameters(): p.requires_grad=True
run_phase(optim.AdamW(model.parameters(),lr=1e-4,weight_decay=1e-4),FINETUNE_EPOCHS,"Fine-tuning")
pd.DataFrame(history).to_csv(HISTORY_PATH,index=False); print("Saved:",CHECKPOINT_PATH.resolve())


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\rickv/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:03<00:00, 13.9MB/s]


Frozen backbone | Epoch 1/3 | train 1.0168/0.5836 | val 0.8275/0.6628
Frozen backbone | Epoch 2/3 | train 0.6736/0.7584 | val 0.7262/0.7002
Frozen backbone | Epoch 3/3 | train 0.6065/0.7820 | val 0.6522/0.7377
Fine-tuning | Epoch 1/7 | train 0.3867/0.8634 | val 0.5058/0.8431
Fine-tuning | Epoch 2/7 | train 0.1134/0.9623 | val 0.3987/0.8525
Fine-tuning | Epoch 3/7 | train 0.0488/0.9874 | val 0.4702/0.8525
Fine-tuning | Epoch 4/7 | train 0.0354/0.9925 | val 0.3153/0.8923
Fine-tuning | Epoch 5/7 | train 0.0259/0.9945 | val 0.4762/0.8782
Fine-tuning | Epoch 6/7 | train 0.0336/0.9910 | val 0.7944/0.8314
Fine-tuning | Epoch 7/7 | train 0.0260/0.9905 | val 0.4301/0.8782
Saved: C:\Users\rickv\School\Jaar3\minor\Challenge\Brain-Tumor\brain_tumors\trained_models\resnet18_best.pth
